# Credit Risk Modeling with Cost-Sensitive Threshold Optimization

## Background & Objective
This project builds a credit default prediction model and focuses on decision-oriented
evaluation under asymmetric misclassification costs. In credit approval systems,
false negatives (approving a default) are typically more costly than false positives.

The goal is not only to build a predictive model, but also to design a decision rule
(classification threshold) that minimizes expected cost.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

## Data Description
The dataset contains borrower-level features and a binary label 'cb_person_default_on_file' indicating default. 

The target variable is highly imbalanced, which motivates the use of ROC-AUC for
model evaluation instead of raw accuracy.

In [26]:
#Read dataset, print first 5 rows
df = pd.read_csv('data/credit_risk_dataset.csv')
print(df.head(5))
print(len(df))

   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                         Y                           3  


In [27]:
#Check for missing values in each column
df.isna().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [28]:
#check the proportion of young people among those with unknown years of employment length
#len(df[(df['person_emp_length'].isna()) & (df['person_age'] <= 25)]) / len(df[df['person_emp_length'].isna()])
#young people (<= 25) can be assumed to have no employment length, howerver, as the portion is only around 0.51, this assumption couldn't cover all the missing values. 
#Thus, we need to find other ways to fill in the missing values, or simply drop these rows. 

#check the maximum age of people with unknown years of employment length
#df[df['person_emp_length'].isna()]['person_age'].max()

## Modeling Approach
A logistic regression model is used as a baseline probabilistic classifier.
A full preprocessing pipeline is constructed using scikit-learn, including
imputation for missing values, feature scaling, and one-hot encoding for
categorical variables.

In [29]:
#define feature columns and target column
target = 'loan_status'
features = [c for c in df.columns if c != target]

#split the data into training and testing sets
X = df[features]
y = df[target]

In [30]:
cat_cols = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade", 
    "cb_person_default_on_file"
]
num_cols = df.columns.difference(cat_cols).tolist()
num_cols.remove('loan_status')

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [31]:
#classifier of cb_person_default_on_file using RandomForestClassifier
classifier = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,          # 70% train, 30% temp
    random_state=41
    #stratify=y
)

X_vald, X_test, y_vald, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,          # 15% val, 15% test
    random_state=41
    #stratify=y_temp
)

loan_vald = X_vald['loan_amnt'].fillna(
    X_train['loan_amnt'].median()
)
interest_rate_vald = X_vald['loan_int_rate'].fillna(
    X_train['loan_int_rate'].median()
)

classifier.fit(X_train, y_train)
#y_vald_pred = classifier.predict(X_vald)
#y_vald_proba = classifier.predict_proba(X_vald)[:, 1]
#print(classification_report(y_test, y_pred))
#y_proba = classifier.predict_proba(X_test)[:, 1]
#print("ROC AUC:", roc_auc_score(y_test, y_proba))


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['cb_person_cred_hist_length',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'person_age',
                                                   'person_emp_length',
                                                   'person_income']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['person_home_ownership',
                                                   'loan_intent', 'loan_grade',
                                                   'cb_person_default_on_file'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

## Regularization and Model Selection
To control model complexity and prevent overfitting, L2 regularization is applied
to the logistic regression model. The regularization strength is treated as a
hyperparameter and selected via cross-validated ROC-AUC on the training set.

This reflects a standard bias–variance tradeoff in statistical learning:
stronger regularization reduces variance but may introduce bias, while weaker
regularization allows more flexibility at the risk of overfitting.

In [32]:
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

gs = GridSearchCV(
    estimator=classifier,      # your Pipeline
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

gs.fit(X_train, y_train)

print("Best C:", gs.best_params_["model__C"])
print("Best CV AUC:", gs.best_score_)
best_clf = gs.best_estimator_

Best C: 10
Best CV AUC: 0.8677690224608303


In [33]:
for mean, std, params in zip(gs.cv_results_['mean_test_score'],
                             gs.cv_results_['std_test_score'],
                             gs.cv_results_['params']):
    print(params, mean, std)

{'model__C': 0.001} 0.8516666050626995 0.008405324932046635
{'model__C': 0.01} 0.8644251656841628 0.008370402349321737
{'model__C': 0.1} 0.8674006968434984 0.009057571162336764
{'model__C': 1} 0.8677392070029427 0.009429510682767563
{'model__C': 10} 0.8677690224608303 0.0094955789218091
{'model__C': 100} 0.8677450879711944 0.009519120530742574


## Cost-Sensitive Decision Rule
Instead of using the default threshold 0.5, a decision threshold based on expected loss is optimized
on a validation set by minimizing total expected loss:

EL = PD(Probability of Default) * loan_amount - (1 - PD) * loan_interest_rate * loan_amount

In [37]:
def threshold_search(y_vald, y_vald_proba, loan_vald, interest_rate_vald):
    #y_vald: sample data, y_vald_proba: model prediction probability

    #cost coefficient for FN and FP
    FP_costcoef = interest_rate_vald/100
    FN_costcoef = 1
    TN_costcoef = interest_rate_vald/100

    threshold, best_loss = None, float("inf")

    for t in np.linspace(-1e6, 1e6, 2000):
        y_hat = ((-(1-y_vald_proba) * loan_vald * TN_costcoef + y_vald_proba * loan_vald * FN_costcoef) >= t).astype(int) #model prediction (w.r.t. threshold t)

        #FP_mask = (y_vald == 0) & (y_hat == 1)
        #FP_cost = (FP_costcoef * loan_vald)[FP_mask].sum()

        FN_mask = (y_vald == 1) & (y_hat == 0)
        FN_cost = FN_costcoef * loan_vald[FN_mask].sum()

        TN_mask = (y_vald == 0) & (y_hat == 0)
        TN_cost = (TN_costcoef * loan_vald)[TN_mask].sum()
        
        loss = FN_cost - TN_cost
        #print(loss)
        if loss < best_loss:
            best_loss, threshold = loss, t
            y_pred_threshold = y_hat
    return y_pred_threshold, threshold, best_loss

y_vald_proba = best_clf.predict_proba(X_vald)[:, 1]
y_pred_threshold, threshold, best_loss = threshold_search(y_vald, y_vald_proba, loan_vald, interest_rate_vald)

print(threshold, best_loss, classification_report(y_vald, y_pred_threshold))#, confusion_matrix(y_vald, y_pred_threshold))

1500.7503751876066 -944565.2549999999               precision    recall  f1-score   support

           0       0.93      0.60      0.73      3785
           1       0.39      0.85      0.53      1102

    accuracy                           0.66      4887
   macro avg       0.66      0.73      0.63      4887
weighted avg       0.81      0.66      0.69      4887



In [35]:
loan_test = X_test['loan_amnt'].fillna(
    X_train['loan_amnt'].median()
)
interest_rate_test = X_test['loan_int_rate'].fillna(
    X_train['loan_int_rate'].median()
)

#FP_costcoef = interest_rate_test
TN_costcoef = interest_rate_test/100
FN_costcoef = 1

y_test_proba = best_clf.predict_proba(X_test)[:, 1]
print("Test AUC:", roc_auc_score(y_test, y_test_proba))
y_test_hat = ((-(1-y_test_proba) * loan_test * TN_costcoef + y_test_proba * loan_test)  >= threshold).astype(int)
print(classification_report(y_test, y_test_hat))

Test AUC: 0.866738211292385
              precision    recall  f1-score   support

           0       0.93      0.60      0.73      3801
           1       0.37      0.84      0.52      1087

    accuracy                           0.65      4888
   macro avg       0.65      0.72      0.62      4888
weighted avg       0.80      0.65      0.68      4888



In [36]:
TN_mask = (y_test == 0) & (y_test_proba < 0.5)
FN_mask = (y_test == 1) & (y_test_proba < 0.5)
profit_test = (TN_costcoef * loan_test)[TN_mask].sum() - FN_costcoef * loan_test[FN_mask].sum()

TN_mask_improved = (y_test == 0) & (y_test_hat == 0)
FN_mask_improved = (y_test == 1) & (y_test_hat == 0)
profit_test_improved = (TN_costcoef * loan_test)[TN_mask_improved].sum() - FN_costcoef * loan_test[FN_mask_improved].sum()

print("original realized profit", profit_test, "improved realized profit", profit_test_improved)
print("Relative improvement:", f"{(profit_test_improved - profit_test) / abs(profit_test):.4%}")

original realized profit 741246.5575000001 improved realized profit 829124.385
Relative improvement: 11.8554%


## Key Insight: Generalization Risk of Decision Thresholds
Although the decision threshold is optimized to minimize weighted cost on the
validation set, the optimized threshold does not always strictly outperform
the default threshold on the held-out test set.

This illustrates the generalization uncertainty of decision-level
hyperparameters and reflects a fundamental bias–variance tradeoff in
cost-sensitive classification.

In [38]:
import os
import pandas as pd
import numpy as np

os.makedirs("artifacts", exist_ok=True)

# predicted default probability on test set
y_test_proba = best_clf.predict_proba(X_test)[:, 1]

# cost coefficients
TN_costcoef_test = interest_rate_test / 100
FN_costcoef = 1

# decision score used by your threshold rule
decision_score_test = (
    -(1 - y_test_proba) * loan_test * TN_costcoef_test
    + y_test_proba * loan_test * FN_costcoef
)

# profit-based decision
# y_hat = 1 means reject / predict default
# y_hat = 0 means approve / predict non-default
y_test_hat = (decision_score_test >= threshold).astype(int)

decision_cases = X_test.copy()

decision_cases["case_id"] = [f"case_{i:04d}" for i in range(len(decision_cases))]
decision_cases["actual_default"] = y_test.values if hasattr(y_test, "values") else y_test
decision_cases["default_prob"] = y_test_proba
decision_cases["decision_score"] = decision_score_test
decision_cases["profit_threshold"] = threshold
decision_cases["model_decision"] = np.where(y_test_hat == 1, "reject", "approve")

# Optional: default 0.5 probability decision for comparison
decision_cases["default_05_decision"] = np.where(y_test_proba >= 0.5, "reject", "approve")

# A short text field for RAG/LLM
decision_cases["explanation_seed"] = decision_cases.apply(
    lambda row: (
        f"Applicant {row['case_id']} was {row['model_decision']} by the profit-based decision layer. "
        f"The predicted default probability is {row['default_prob']:.3f}. "
        f"The decision score is {row['decision_score']:.2f}, compared with the optimized threshold "
        f"{row['profit_threshold']:.2f}. "
        f"The default 0.5 probability rule would make decision: {row['default_05_decision']}."
    ),
    axis=1
)

decision_cases.to_csv("artifacts/decision_cases.csv", index=False)